In [2]:
#https://blog.brunk.io/posts/similarity-search-with-duckdb

In [3]:
!pip install duckdb transformers FlagEmbedding polars pyarrow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 7.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.0/149.0 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 68.1 MB/s eta 0:00:00
  Created wheel for FlagEmbedding: filename=FlagEmbedding-1.3.5-py3-none-any.whl size=233746 sha256=5cc00510521517d8fa48417d087696e65694095a0c6b407f9dce7d093912274b
  Stored in directory: /root/.cache/pip/wheels/b2/1f/f6/78f862bb80cb959cc9960b7c4e2d1f702b1bc0e79d19b5f124
  Created wheel for warc3-wet-clueweb09: filename=warc3_wet_clueweb09-0.2.5-py3-none-any.whl size=18919 sha256=90dd7a121669df445e9a49f4b0068c93deb20968ef451b05fcf4bf9278b2ff70
  Stored in directory: /root/.c

In [4]:
import duckdb
from FlagEmbedding import BGEM3FlagModel
import torch


con = duckdb.connect()
con.install_extension("vss")
con.load_extension("vss")

sql="""SET GLOBAL hnsw_enable_experimental_persistence = true;"""

con.execute(sql)


In [5]:
qry = """FROM 'hf://datasets/wikimedia/wikipedia/20231101.en/*.parquet'
SELECT count(*) AS count;"""

con.execute(qry).fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,count
0,6407814


In [6]:
qry = """FROM 'hf://datasets/wikimedia/wikipedia/20231101.en/*.parquet'
SELECT * LIMIT 10;"""

con.execute(qry).fetchdf()

,id,url,title,text
0,12,https://en.wikipedia.org/wiki/Anarchism,Anarchism,Anarchism is a political philosophy and moveme...
1,39,https://en.wikipedia.org/wiki/Albedo,Albedo,Albedo (; ) is the fraction of sunlight that i...
2,290,https://en.wikipedia.org/wiki/A,A,"A, or a, is the first letter and the first vow..."
3,303,https://en.wikipedia.org/wiki/Alabama,Alabama,Alabama () is a state in the Southeastern regi...
4,305,https://en.wikipedia.org/wiki/Achilles,Achilles,"In Greek mythology, Achilles ( ) or Achilleus ..."
5,307,https://en.wikipedia.org/wiki/Abraham%20Lincoln,Abraham Lincoln,"Abraham Lincoln ( ; February 12, 1809 – April ..."
6,308,https://en.wikipedia.org/wiki/Aristotle,Aristotle,"Aristotle (; Aristotélēs, ; 384–322 BC) was a..."
7,309,https://en.wikipedia.org/wiki/An%20American%20...,An American in Paris,An American in Paris is a jazz-influenced symp...
8,316,https://en.wikipedia.org/wiki/Academy%20Award%...,Academy Award for Best Production Design,The Academy Award for Best Production Design r...
9,324,https://en.wikipedia.org/wiki/Academy%20Awards,Academy Awards,"The Academy Awards, mainly known as the Oscars..."


In [7]:
device = "cpu"
# use a GPU if available to speed up the embedding computation
if torch.cuda.is_available(): device = "cuda" # Nvidia GPU
elif torch.backends.mps.is_available(): device = "mps" # Apple silicon GPU

model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True, device=device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

...

onnx/model.onnx_data:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

onnx/tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

sparse_linear.pt:   0%|          | 0.00/3.52k [00:00<?, ?B/s]

In [8]:
print(device)

cuda


In [9]:
#Aside
queries = ["What is BGE M3?", "What is DuckDB?"]
documents = [
    "BGE M3 is an embedding model supporting dense retrieval, lexical matching and multi-vector interaction.",
    "DuckDB is a fast in-process analytical database. It supports a feature-rich SQL dialect complemented with deep integrations into client APIs",
]

query_embeddings = model.encode(queries)["dense_vecs"]
document_embeddings = model.encode(documents)["dense_vecs"]

similarity = query_embeddings @ document_embeddings.T
similarity


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


array([[0.626 , 0.1917],
       [0.3362, 0.7314]], dtype=float16)

In [10]:
qry = """CREATE TABLE wikipedia AS
FROM 'hf://datasets/wikimedia/wikipedia/20231101.en/*.parquet'
SELECT * limit 200;"""

con.execute(qry)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [11]:
con.execute("SELECT * FROM wikipedia LIMIT 10").fetch_df()

,id,url,title,text
0,12,https://en.wikipedia.org/wiki/Anarchism,Anarchism,Anarchism is a political philosophy and moveme...
1,39,https://en.wikipedia.org/wiki/Albedo,Albedo,Albedo (; ) is the fraction of sunlight that i...
2,290,https://en.wikipedia.org/wiki/A,A,"A, or a, is the first letter and the first vow..."
3,303,https://en.wikipedia.org/wiki/Alabama,Alabama,Alabama () is a state in the Southeastern regi...
4,305,https://en.wikipedia.org/wiki/Achilles,Achilles,"In Greek mythology, Achilles ( ) or Achilleus ..."
5,307,https://en.wikipedia.org/wiki/Abraham%20Lincoln,Abraham Lincoln,"Abraham Lincoln ( ; February 12, 1809 – April ..."
6,308,https://en.wikipedia.org/wiki/Aristotle,Aristotle,"Aristotle (; Aristotélēs, ; 384–322 BC) was a..."
7,309,https://en.wikipedia.org/wiki/An%20American%20...,An American in Paris,An American in Paris is a jazz-influenced symp...
8,316,https://en.wikipedia.org/wiki/Academy%20Award%...,Academy Award for Best Production Design,The Academy Award for Best Production Design r...
9,324,https://en.wikipedia.org/wiki/Academy%20Awards,Academy Awards,"The Academy Awards, mainly known as the Oscars..."


In [12]:
qry = """CREATE TABLE embeddings(
     doc_id VARCHAR,
     embedding FLOAT[1024]
);"""

con.execute(qry)

In [13]:
import pyarrow as pa
import numpy as np

reader = con.execute(
    "FROM wikipedia SELECT id, text WHERE id NOT IN (FROM embeddings select doc_id);"
).fetch_record_batch(100)

for batch in reader: # 100 records per batch
    # 4 records per GPU batch
    embeddings = model.encode(batch["text"].tolist(), batch_size=4)["dense_vecs"].astype(np.float32)
    # add our embeddings to the batch
    # We will now directly insert into the table instead of using a temporary Arrow table
    for i in range(len(batch)):
        con.cursor().execute(
            "INSERT INTO embeddings (doc_id, embedding) VALUES (?, ?);",
            [batch["id"][i].as_py(), embeddings[i].tolist()]
        )

Inference Embeddings: 100%|██████████| 25/25 [00:01<00:00, 13.59it/s]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [14]:
sql = """FROM wikipedia JOIN embeddings ON (wikipedia.id = embeddings.doc_id)
SELECT *
LIMIT 5;"""

con.execute(sql).fetch_df()

,id,url,title,text,doc_id,embedding
0,12,https://en.wikipedia.org/wiki/Anarchism,Anarchism,Anarchism is a political philosophy and moveme...,12,"[0.02633667, -0.0013446808, 0.0079574585, 0.00..."
1,39,https://en.wikipedia.org/wiki/Albedo,Albedo,Albedo (; ) is the fraction of sunlight that i...,39,"[-0.028518677, 0.009605408, -0.04446411, -0.02..."
2,290,https://en.wikipedia.org/wiki/A,A,"A, or a, is the first letter and the first vow...",290,"[-0.00059890747, 0.012763977, -0.015975952, -0..."
3,303,https://en.wikipedia.org/wiki/Alabama,Alabama,Alabama () is a state in the Southeastern regi...,303,"[-0.0011062622, -0.0032577515, -0.027740479, -..."
4,305,https://en.wikipedia.org/wiki/Achilles,Achilles,"In Greek mythology, Achilles ( ) or Achilleus ...",305,"[-0.00390625, 0.0021438599, -0.035064697, 0.01..."


In [15]:

from duckdb.typing import VARCHAR

def embed(sentence: str) -> np.ndarray:
    return model.encode(sentence)['dense_vecs']

con.create_function("embed", embed, [VARCHAR], 'FLOAT[1024]')


qry = "SELECT embed('Who was the first human on the moon?') AS query_embedding;"
con.execute(qry).fetch_df()

,query_embedding
0,"[0.011383057, 0.010261536, -0.058807373, -0.04..."


In [16]:
def search(q: str):
    return con.execute("""
        FROM embeddings JOIN wikipedia ON (wikipedia.id = embeddings.doc_id)
        SELECT wikipedia.id, title, array_inner_product(embedding, embed($q)) AS similarity
        ORDER BY similarity DESC
        LIMIT 5""",
        {"q": q}
    ).pl()

search('Who was the first human on the moon?')

id,title,similarity
str,str,f32
"""662""","""Apollo 11""",0.677635
"""663""","""Apollo 8""",0.612966
"""664""","""Astronaut""",0.45167
"""728""","""List of anthropologists""",0.43085
"""784""","""Alfred Korzybski""",0.420688


In [17]:
qry = """CREATE INDEX vec_idx ON embeddings USING HNSW (embedding)
WITH (metric = 'cosine');"""
con.execute(qry)

In [19]:
q = "Who was the first person on the moon?"

qry="""WITH top_k AS (
    FROM embeddings SELECT *
    ORDER BY array_inner_product(embedding, embed($q)) DESC
    LIMIT 5)
FROM top_k JOIN wikipedia ON (wikipedia.id = top_k.doc_id)
SELECT wikipedia.id, title, array_inner_product(embedding, embed($q)) AS similarity
ORDER BY similarity DESC"""

con.execute(qry, {"q": q}).fetch_df()

,id,title,similarity
0,662,Apollo 11,0.673158
1,663,Apollo 8,0.610035
2,664,Astronaut,0.470620
3,728,List of anthropologists,0.447238
4,851,Alfred Nobel,0.422824
